In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, itertools, pathlib
warnings.filterwarnings('ignore')
pathlib.Path('figs').mkdir(exist_ok=True)
_TAG = 'nb3'
_fig_counter = itertools.count(1)
def _save_show(*a, **k):
    import matplotlib.pyplot as _plt
    for _n in _plt.get_fignums():
        _plt.figure(_n).savefig('figs/%s_fig%02d.png' % (_TAG, next(_fig_counter)), dpi=140, bbox_inches='tight')
    _plt.close('all')
plt.show = _save_show


# Characterizing Orbital Perturbations via Physics-Informed Neural Networks  
## Notebook 3: First Physics-Informed Model for Sparse Trajectory Reconstruction and Hidden Parameter Recovery

This notebook is the first true modeling notebook in the project.

Notebook 1 established the controlled synthetic dataset and the hidden perturbation parameter.  
Notebook 2 established a unified data schema and showed how real orbital datasets can be imported.

Now the project asks its first real inverse-problem machine learning question:

> **Can a physics-informed neural network reconstruct an orbital trajectory from sparse observations while also learning a hidden perturbation strength?**

### Goals for this notebook
1. Load the synthetic sparse-observation dataset from Notebook 1, with a fallback that can regenerate it if needed
2. Build a first neural network that maps time to orbital state
3. Add a **physics loss** so the prediction is constrained by orbital dynamics
4. Treat the drag-like perturbation strength as a **learnable parameter**
5. Compare the learned parameter against the known truth and against the controlled inverse baseline from Notebook 1
6. Check how the same workflow could later extend to the standardized real datasets prepared in Notebook 2

This is the notebook where the project stops being only a simulation pipeline and becomes a real scientific modeling workflow!

A purely data-driven network could memorize sparse points.  

A physics-informed network is asked to do something stronger:
- fit the observations
- obey the equations of motion
- discover hidden physical information from the observed trajectory

That is the central scientific claim of the project.

## 1. Where this notebook fits in the project

The current project sequence is now:

1. **Notebook 1**  
   Build a controlled synthetic orbit dataset with a known hidden perturbation parameter

2. **Notebook 2**  
   Define a unified observation schema and begin importing real orbital datasets

3. **Notebook 3**  
   Train the first physics-informed model on the controlled synthetic inverse-problem case

The project should not jump straight to real data before proving that the modeling framework works on a case where:
- the ground truth is known
- the hidden parameter is known
- the noise level is known
- the observation sparsity is known

That is why this notebook still focuses on the synthetic branch.

### Important modeling choice in this notebook
A fully continuous PINN usually computes time derivatives through automatic differentiation.

For a first project notebook, a very good stepping stone is a **physics-constrained neural network with a discrete physics residual**:
- the network predicts the state on a time grid
- finite differences approximate the time derivatives
- the residual of the orbital equations is penalized during training

## 2. Environment setup

This notebook uses:
- `numpy`
- `pandas`
- `matplotlib`
- `scipy`
- `torch`
- `pathlib`

The notebook is written so that it can:
- load the CSV files created in Notebook 1 if they are present
- otherwise regenerate the same synthetic dataset directly

That makes the notebook more robust for a student workflow, especially if files were not saved in the current runtime!

In [2]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

import torch
import torch.nn as nn

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_DIR = Path(".")


Using device: cpu


## 3. Load the synthetic dataset, or regenerate it if the CSV files are missing

Notebook 1 should have created three files:
- `synthetic_dense_truth.csv`
- `synthetic_sparse_observations.csv`
- `synthetic_metadata.csv`

Because students often work across multiple runtimes, this notebook includes a fallback that can regenerate the same synthetic orbit setup. That way, the notebook is still self-contained.

### Reminder of the synthetic case
The controlled hidden parameter is a simple drag-like coefficient.  
The forward dynamics include:
- two-body gravity
- Earth $J_2$
- a simple altitude-dependent drag-like term

The model will later try to learn that hidden drag coefficient from sparse noisy observations.

In [3]:
# Earth constants in km, s units
MU_EARTH = 398600.4418
R_EARTH = 6378.1363
J2_EARTH = 1.08262668e-3
DEFAULT_DRAG_COEFF = 2e-5


def norm(vec):
    return np.linalg.norm(vec)


def two_body_acceleration(r, mu=MU_EARTH):
    r_norm = norm(r)
    return -mu * r / (r_norm**3)


def j2_acceleration(r, mu=MU_EARTH, R=R_EARTH, J2=J2_EARTH):
    x, y, z = r
    r_norm = norm(r)
    z2 = z**2
    r2 = r_norm**2

    factor = 1.5 * J2 * mu * (R**2) / (r_norm**5)
    common = 5.0 * z2 / r2

    ax = factor * x * (common - 1.0)
    ay = factor * y * (common - 1.0)
    az = factor * z * (common - 3.0)
    return np.array([ax, ay, az])


def simple_drag_acceleration(r, v, drag_coeff=DEFAULT_DRAG_COEFF):
    H = 60.0
    altitude = max(norm(r) - R_EARTH, 0.0)
    density_factor = np.exp(-altitude / H)
    speed = norm(v)
    return -drag_coeff * density_factor * speed * v


def orbital_dynamics(t, state, use_j2=True, use_drag=False, drag_coeff=DEFAULT_DRAG_COEFF):
    r = state[:3]
    v = state[3:]

    a = two_body_acceleration(r)
    if use_j2:
        a = a + j2_acceleration(r)
    if use_drag:
        a = a + simple_drag_acceleration(r, v, drag_coeff=drag_coeff)

    return np.concatenate([v, a])


def circular_velocity(radius_km, mu=MU_EARTH):
    return np.sqrt(mu / radius_km)


def regenerate_synthetic_dataset(
    altitude_km=500.0,
    inclination_deg=35.0,
    num_orbits=8,
    n_eval=2500,
    observe_every=25,
    position_noise_std_km=1.0,
    velocity_noise_std_kms=0.002,
    true_drag_coeff=DEFAULT_DRAG_COEFF,
    seed=42,
):
    radius0 = R_EARTH + altitude_km
    r0 = np.array([radius0, 0.0, 0.0])

    v_circ = circular_velocity(radius0)
    inc = np.deg2rad(inclination_deg)
    v0 = np.array([
        0.0,
        v_circ * np.cos(inc),
        v_circ * np.sin(inc),
    ])
    state0 = np.concatenate([r0, v0])

    orbital_period_est = 2 * np.pi * np.sqrt(radius0**3 / MU_EARTH)
    t_final = num_orbits * orbital_period_est
    t_eval = np.linspace(0.0, t_final, n_eval)

    sol = solve_ivp(
        lambda t, y: orbital_dynamics(t, y, use_j2=True, use_drag=True, drag_coeff=true_drag_coeff),
        (0.0, t_final),
        state0,
        t_eval=t_eval,
        rtol=1e-9,
        atol=1e-9,
    )

    truth_t = sol.t
    truth_state = sol.y.T

    obs_t = truth_t[::observe_every]
    obs_state_clean = truth_state[::observe_every]

    rng = np.random.default_rng(seed)
    obs_state_noisy = obs_state_clean.copy()
    obs_state_noisy[:, :3] += rng.normal(0.0, position_noise_std_km, size=obs_state_noisy[:, :3].shape)
    obs_state_noisy[:, 3:] += rng.normal(0.0, velocity_noise_std_kms, size=obs_state_noisy[:, 3:].shape)

    dense_df = pd.DataFrame({
        "t_sec": truth_t,
        "x_km": truth_state[:, 0],
        "y_km": truth_state[:, 1],
        "z_km": truth_state[:, 2],
        "vx_kms": truth_state[:, 3],
        "vy_kms": truth_state[:, 4],
        "vz_kms": truth_state[:, 5],
    })

    sparse_df = pd.DataFrame({
        "t_sec": obs_t,
        "x_km": obs_state_noisy[:, 0],
        "y_km": obs_state_noisy[:, 1],
        "z_km": obs_state_noisy[:, 2],
        "vx_kms": obs_state_noisy[:, 3],
        "vy_kms": obs_state_noisy[:, 4],
        "vz_kms": obs_state_noisy[:, 5],
    })

    metadata_df = pd.DataFrame({
        "parameter": [
            "true_drag_coeff",
            "use_j2",
            "position_noise_std_km",
            "velocity_noise_std_kms",
            "observe_every",
            "altitude_km",
            "inclination_deg",
            "n_dense_points",
            "n_sparse_points",
        ],
        "value": [
            true_drag_coeff,
            True,
            position_noise_std_km,
            velocity_noise_std_kms,
            observe_every,
            altitude_km,
            inclination_deg,
            len(dense_df),
            len(sparse_df),
        ],
    })

    return dense_df, sparse_df, metadata_df


def load_or_regenerate_synthetic_files(data_dir=DATA_DIR):
    dense_path = data_dir / "synthetic_dense_truth.csv"
    sparse_path = data_dir / "synthetic_sparse_observations.csv"
    metadata_path = data_dir / "synthetic_metadata.csv"

    if dense_path.exists() and sparse_path.exists() and metadata_path.exists():
        print("Loading synthetic dataset from Notebook 1 CSV files.")
        dense_df = pd.read_csv(dense_path)
        sparse_df = pd.read_csv(sparse_path)
        metadata_df = pd.read_csv(metadata_path)
        source_mode = "loaded_from_csv"
    else:
        print("Synthetic CSV files were not found. Regenerating the Notebook 1 dataset in this notebook.")
        dense_df, sparse_df, metadata_df = regenerate_synthetic_dataset()
        source_mode = "regenerated_in_notebook_3"

    return dense_df, sparse_df, metadata_df, source_mode


synthetic_dense, synthetic_sparse, synthetic_metadata, source_mode = load_or_regenerate_synthetic_files(DATA_DIR)

print("Source mode:", source_mode)
print("Dense shape:", synthetic_dense.shape)
print("Sparse shape:", synthetic_sparse.shape)
display(synthetic_metadata)


Loading synthetic dataset from Notebook 1 CSV files.
Source mode: loaded_from_csv
Dense shape: (2500, 7)
Sparse shape: (72, 7)


,parameter,value
0,true_drag_coeff,3.245e-05
1,use_j2,True
2,position_noise_std_km,0.5
3,velocity_noise_std_kms,0.002
4,observe_every,35
5,altitude_km,500.0
6,inclination_deg,35.0
7,n_dense_points,2500
8,n_sparse_points,72


## 4. Inspect the sparse and dense trajectories

Before training any model, it is worth visually checking the dataset.

The dense trajectory is the hidden reference truth.  
The sparse trajectory is what the model actually sees.

If the sparse points look too dense, the inverse problem becomes too easy.  
If they are too sparse, the reconstruction problem becomes much harder.

That balance matters.

In [4]:
truth_t = synthetic_dense["t_sec"].to_numpy()
truth_state = synthetic_dense[["x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms"]].to_numpy()

obs_t = synthetic_sparse["t_sec"].to_numpy()
obs_state = synthetic_sparse[["x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms"]].to_numpy()

plt.figure(figsize=(7, 7))
plt.plot(truth_state[:, 0], truth_state[:, 1], label="Dense truth trajectory")
plt.scatter(obs_state[:, 0], obs_state[:, 1], s=20, label="Sparse noisy observations")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title("Synthetic orbit: dense truth vs sparse observations")
plt.axis("equal")
plt.legend()
plt.show()

plt.figure()
plt.plot(truth_t / 3600.0, np.linalg.norm(truth_state[:, :3], axis=1), label="Dense truth radius")
plt.scatter(obs_t / 3600.0, np.linalg.norm(obs_state[:, :3], axis=1), s=20, label="Sparse noisy radius")
plt.xlabel("Time [hours]")
plt.ylabel("Radius [km]")
plt.title("Radius over time")
plt.legend()
plt.show()


## 5. Optional continuity check with Notebook 2 outputs

Notebook 2 introduced a standardized data schema for multiple datasets.

This notebook still trains on the synthetic controlled case, but we can already check whether any standardized files from Notebook 2 are present.  
That keeps the student aware that the project is building toward a multi-source workflow rather than a one-off simulation.

The expected standardized files are:
- `notebook2_synthetic_standardized.csv`
- `notebook2_celestrak_iss_standardized.csv`
- `notebook2_horizons_moon_standardized.csv`

In [5]:
standardized_candidates = {
    "synthetic_standardized": DATA_DIR / "notebook2_synthetic_standardized.csv",
    "celestrak_standardized": DATA_DIR / "notebook2_celestrak_iss_standardized.csv",
    "horizons_standardized": DATA_DIR / "notebook2_horizons_moon_standardized.csv",
}

for label, path in standardized_candidates.items():
    print(f"{label}: {path.exists()} -> {path}")


synthetic_standardized: False -> notebook2_synthetic_standardized.csv
celestrak_standardized: True -> notebook2_celestrak_iss_standardized.csv
horizons_standardized: True -> notebook2_horizons_moon_standardized.csv


## 6. Prepare the training representation

A neural network usually trains more stably if:
- time is scaled to a compact interval such as $[0, 1]$
- state variables are normalized

We will use:
- **dense truth** for evaluation and for building the collocation time grid
- **sparse observations** for the data-fitting loss

### Important conceptual split
The model is trained against two types of information:

1. **Data information**  
   Sparse observations tell the model what the trajectory looked like at observed times

2. **Physics information**  
   The equations of motion tell the model how trajectories should evolve between those sparse times

That split is what makes the approach ***physics-informed***.

In [6]:
true_drag_coeff = None
if "true_drag_coeff" in synthetic_metadata["parameter"].values:
    true_drag_coeff = float(
        synthetic_metadata.loc[synthetic_metadata["parameter"] == "true_drag_coeff", "value"].iloc[0]
    )

state_mean = truth_state.mean(axis=0, keepdims=True)
state_std = truth_state.std(axis=0, keepdims=True)
state_std[state_std < 1e-12] = 1.0

t0 = truth_t.min()
t_scale = truth_t.max() - truth_t.min()

truth_t_norm = ((truth_t - t0) / t_scale).reshape(-1, 1)
obs_t_norm = ((obs_t - t0) / t_scale).reshape(-1, 1)

truth_state_norm = (truth_state - state_mean) / state_std
obs_state_norm = (obs_state - state_mean) / state_std

initial_state = truth_state[0:1]
initial_state_norm = (initial_state - state_mean) / state_std

COLLOCATION_POINTS = 160
colloc_idx = np.linspace(0, len(truth_t) - 1, COLLOCATION_POINTS).astype(int)
colloc_idx = np.unique(colloc_idx)
colloc_t = truth_t[colloc_idx]
colloc_t_norm = truth_t_norm[colloc_idx]

reference_speed = float(np.mean(np.linalg.norm(truth_state[:, 3:], axis=1)))
reference_acceleration = float(MU_EARTH / (np.mean(np.linalg.norm(truth_state[:, :3], axis=1)) ** 2))

print("True drag coefficient:", true_drag_coeff)
print("Time span [hours]:", t_scale / 3600.0)
print("Number of sparse observations:", len(obs_t))
print("Number of collocation points:", len(colloc_t))
print("Reference speed [km/s]:", reference_speed)
print("Reference acceleration [km/s^2]:", reference_acceleration)


True drag coefficient: 3.245e-05
Time span [hours]: 12.615504804204324
Number of sparse observations: 72
Number of collocation points: 160
Reference speed [km/s]: 7.636162054101145
Reference acceleration [km/s^2]: 0.008518092861314094


## 7. Convert the training arrays into Torch tensors

We will store three main tensor groups:

- **observation tensors** for the sparse data loss
- **collocation tensors** for the physics residual
- **evaluation tensors** for plotting the full reconstructed trajectory

The hidden drag coefficient will be learned as a separate parameter.

In [7]:
obs_t_tensor = torch.tensor(obs_t_norm, dtype=torch.float32, device=device)
obs_y_tensor = torch.tensor(obs_state_norm, dtype=torch.float32, device=device)

colloc_t_tensor = torch.tensor(colloc_t_norm, dtype=torch.float32, device=device)
colloc_t_phys_tensor = torch.tensor(colloc_t.reshape(-1, 1), dtype=torch.float32, device=device)

eval_t_tensor = torch.tensor(truth_t_norm, dtype=torch.float32, device=device)
eval_t_phys_tensor = torch.tensor(truth_t.reshape(-1, 1), dtype=torch.float32, device=device)

state_mean_tensor = torch.tensor(state_mean, dtype=torch.float32, device=device)
state_std_tensor = torch.tensor(state_std, dtype=torch.float32, device=device)
initial_state_norm_tensor = torch.tensor(initial_state_norm, dtype=torch.float32, device=device)

print("Observation tensor shape:", obs_t_tensor.shape, obs_y_tensor.shape)
print("Collocation tensor shape:", colloc_t_tensor.shape)
print("Evaluation tensor shape:", eval_t_tensor.shape)


Observation tensor shape: torch.Size([72, 1]) torch.Size([72, 6])
Collocation tensor shape: torch.Size([160, 1])
Evaluation tensor shape: torch.Size([2500, 1])


## 8. Define the physics model in Torch

The neural network will predict the 6-dimensional orbital state:

$[x, y, z, v_x, v_y, v_z]$

The physics residual will compare the network prediction against the first-order orbital system:

$\dot{\mathbf{r}} = \mathbf{v}$

$\dot{\mathbf{v}} = \mathbf{a}_{2body} + \mathbf{a}_{J_2} + \mathbf{a}_{drag}$

### Why define the dynamics again in Torch?
Because the loss function must be differentiable with respect to:
- the network weights
- the hidden drag coefficient

So the orbital acceleration model must be written using Torch operations.

In [8]:
def torch_norm(x):
    return torch.sqrt(torch.sum(x * x, dim=1, keepdim=True) + 1e-12)


def two_body_acceleration_torch(r, mu=MU_EARTH):
    r_norm = torch_norm(r)
    return -mu * r / (r_norm**3)


def j2_acceleration_torch(r, mu=MU_EARTH, R=R_EARTH, J2=J2_EARTH):
    x = r[:, 0:1]
    y = r[:, 1:2]
    z = r[:, 2:3]
    r_norm = torch_norm(r)
    z2 = z * z
    r2 = r_norm * r_norm

    factor = 1.5 * J2 * mu * (R**2) / (r_norm**5)
    common = 5.0 * z2 / r2

    ax = factor * x * (common - 1.0)
    ay = factor * y * (common - 1.0)
    az = factor * z * (common - 3.0)
    return torch.cat([ax, ay, az], dim=1)


def simple_drag_acceleration_torch(r, v, drag_coeff):
    H = 60.0
    altitude = torch.clamp(torch_norm(r) - R_EARTH, min=0.0)
    density_factor = torch.exp(-altitude / H)
    speed = torch_norm(v)
    return -drag_coeff * density_factor * speed * v


def denormalize_state(state_norm):
    return state_norm * state_std_tensor + state_mean_tensor


def finite_difference_derivative(y, t_phys):
    # Approximate dy/dt on a sorted time grid using central differences.
    # This is the discrete physics ingredient used in this first model.
    dy_dt = torch.zeros_like(y)
    dy_dt[1:-1] = (y[2:] - y[:-2]) / (t_phys[2:] - t_phys[:-2])
    dy_dt[0] = (y[1] - y[0]) / (t_phys[1] - t_phys[0])
    dy_dt[-1] = (y[-1] - y[-2]) / (t_phys[-1] - t_phys[-2])
    return dy_dt


## 9. Define the neural network and the learnable perturbation parameter

The input is one scalar: normalized time.  
The output is the 6-dimensional normalized state.

### Hidden parameter design
The drag coefficient must remain positive.  
A convenient way to enforce that is to learn its logarithm and exponentiate it.

So the trainable scalar is:
- `log_drag_param`

and the physical drag coefficient used in the dynamics is:
- `exp(log_drag_param)`

In [9]:
class OrbitMLP(nn.Module):
    def __init__(self, input_dim=1, output_dim=6, hidden_width=64, hidden_depth=3):
        super().__init__()

        layers = []
        in_dim = input_dim
        for _ in range(hidden_depth):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(nn.Tanh())
            in_dim = hidden_width
        layers.append(nn.Linear(in_dim, output_dim))

        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)


config = {
    "hidden_width": 64,
    "hidden_depth": 3,
    "learning_rate": 2e-3,
    "epochs": 600,
    "print_every": 50,
    "data_weight": 10.0,
    "ic_weight": 30.0,
    "physics_weight": 1.0,
    "initial_drag_guess": 1e-5,
}

model = OrbitMLP(
    hidden_width=config["hidden_width"],
    hidden_depth=config["hidden_depth"],
).to(device)

log_drag_param = nn.Parameter(torch.tensor(math.log(config["initial_drag_guess"]), dtype=torch.float32, device=device))

optimizer = torch.optim.Adam(list(model.parameters()) + [log_drag_param], lr=config["learning_rate"])

print(model)
print("Initial drag guess:", float(torch.exp(log_drag_param).detach().cpu()))


OrbitMLP(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=6, bias=True)
  )
)
Initial drag guess: 1.0000003385357559e-05


## 10. Build the composite loss function

The training loss has three parts.

### 1. Data loss
This forces the network to match the sparse observed states.

### 2. Initial-condition loss
This anchors the trajectory at the initial state.  
That is appropriate for this first controlled inverse-problem notebook because the initial state is known from the synthetic setup.

### 3. Physics loss
This compares the predicted derivatives against the orbital dynamics model.

The physics loss is built from two residual blocks:
- kinematic residual: $\dot{\mathbf{r}} - \mathbf{v}$
- dynamic residual: $\dot{\mathbf{v}} - \mathbf{a}(\mathbf{r}, \mathbf{v})$

### Why scale the residuals?
Velocity and acceleration live on different numerical scales.  
If we do not scale them, one part of the physics loss can dominate the other for purely numerical reasons.

So the residuals are divided by representative speed and acceleration scales.

In [10]:
def compute_losses(model, log_drag_param):
    pred_obs_state_norm = model(obs_t_tensor)
    data_loss = torch.mean((pred_obs_state_norm - obs_y_tensor) ** 2)

    pred_initial_norm = model(torch.zeros((1, 1), dtype=torch.float32, device=device))
    ic_loss = torch.mean((pred_initial_norm - initial_state_norm_tensor) ** 2)

    pred_colloc_state_norm = model(colloc_t_tensor)
    pred_colloc_state = denormalize_state(pred_colloc_state_norm)
    pred_colloc_state_dot = finite_difference_derivative(pred_colloc_state, colloc_t_phys_tensor)

    r_pred = pred_colloc_state[:, :3]
    v_pred = pred_colloc_state[:, 3:]

    drag_coeff = torch.exp(log_drag_param)
    accel_pred = (
        two_body_acceleration_torch(r_pred)
        + j2_acceleration_torch(r_pred)
        + simple_drag_acceleration_torch(r_pred, v_pred, drag_coeff)
    )

    kinematic_residual = (pred_colloc_state_dot[:, :3] - v_pred) / reference_speed
    dynamic_residual = (pred_colloc_state_dot[:, 3:] - accel_pred) / reference_acceleration

    physics_loss = torch.mean(kinematic_residual**2) + torch.mean(dynamic_residual**2)

    total_loss = (
        config["data_weight"] * data_loss
        + config["ic_weight"] * ic_loss
        + config["physics_weight"] * physics_loss
    )

    return {
        "total_loss": total_loss,
        "data_loss": data_loss,
        "ic_loss": ic_loss,
        "physics_loss": physics_loss,
        "drag_coeff": drag_coeff,
    }


## 11. Train the first physics-informed model

This is the first real learning step of the project.

The model is being asked to do two things at once:
- reconstruct the sparse observed state trajectory
- find a trajectory that also obeys the orbital equations

If the hidden drag coefficient converges toward the true value, that is already a meaningful proof-of-concept result.

### Runtime note
If the training takes too long, reduce:
- `epochs`
- `hidden_width`
- `COLLOCATION_POINTS`

If the model is unstable, the first parameters to tune are:
- `learning_rate`
- `data_weight`
- `ic_weight`
- `physics_weight`

In [11]:
history = {
    "epoch": [],
    "total_loss": [],
    "data_loss": [],
    "ic_loss": [],
    "physics_loss": [],
    "drag_coeff": [],
}

for epoch in range(1, config["epochs"] + 1):
    optimizer.zero_grad()
    losses = compute_losses(model, log_drag_param)
    losses["total_loss"].backward()
    torch.nn.utils.clip_grad_norm_(list(model.parameters()) + [log_drag_param], max_norm=1.0)
    optimizer.step()

    history["epoch"].append(epoch)
    history["total_loss"].append(float(losses["total_loss"].detach().cpu()))
    history["data_loss"].append(float(losses["data_loss"].detach().cpu()))
    history["ic_loss"].append(float(losses["ic_loss"].detach().cpu()))
    history["physics_loss"].append(float(losses["physics_loss"].detach().cpu()))
    history["drag_coeff"].append(float(losses["drag_coeff"].detach().cpu()))

    if epoch % config["print_every"] == 0 or epoch == 1:
        print(
            f"Epoch {epoch:4d} | "
            f"total={history['total_loss'][-1]:.6f} | "
            f"data={history['data_loss'][-1]:.6f} | "
            f"ic={history['ic_loss'][-1]:.6f} | "
            f"physics={history['physics_loss'][-1]:.6f} | "
            f"drag={history['drag_coeff'][-1]:.8f}"
        )

history_df = pd.DataFrame(history)
history_df.tail()


Epoch    1 | total=6186.005859 | data=1.011076 | ic=1.046768 | physics=6144.492188 | drag=0.00001000


Epoch   50 | total=14.319318 | data=1.334443 | ic=0.004694 | physics=0.834075 | drag=0.00001043


Epoch  100 | total=13.137844 | data=1.219302 | ic=0.000076 | physics=0.942538 | drag=0.00001044


Epoch  150 | total=12.992242 | data=1.198182 | ic=0.001738 | physics=0.958284 | drag=0.00001058


Epoch  200 | total=12.886647 | data=1.194123 | ic=0.000040 | physics=0.944224 | drag=0.00001069


...

Epoch  450 | total=12.767634 | data=1.166688 | ic=0.003553 | physics=0.994168 | drag=0.00001127


Epoch  500 | total=12.671996 | data=1.169533 | ic=0.000774 | physics=0.953443 | drag=0.00001140


Epoch  550 | total=12.715155 | data=1.163740 | ic=0.003181 | physics=0.982328 | drag=0.00001156


Epoch  600 | total=12.629952 | data=1.165060 | ic=0.000912 | physics=0.951999 | drag=0.00001181


,epoch,total_loss,data_loss,ic_loss,physics_loss,drag_coeff
595,596,12.632698,1.161633,0.000945,0.988024,0.000012
596,597,12.637251,1.162559,0.001368,0.970635,0.000012
597,598,12.630211,1.165632,0.001102,0.940821,0.000012
598,599,12.640091,1.168074,0.001154,0.924732,0.000012
599,600,12.629952,1.165060,0.000912,0.951999,0.000012


## 12. Inspect the training history

The two most important things to watch are:
- whether the total loss is decreasing
- whether the learned drag coefficient is moving toward a stable value

A decreasing loss alone is not enough.  
The learned parameter must also make physical sense.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_df["epoch"], history_df["total_loss"], label="Total loss")
axes[0].plot(history_df["epoch"], history_df["data_loss"], label="Data loss")
axes[0].plot(history_df["epoch"], history_df["ic_loss"], label="IC loss")
axes[0].plot(history_df["epoch"], history_df["physics_loss"], label="Physics loss")
axes[0].set_yscale("log")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training history")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["drag_coeff"], label="Learned drag coefficient")
if true_drag_coeff is not None:
    axes[1].axhline(true_drag_coeff, linestyle="--", label="True drag coefficient")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Drag coefficient")
axes[1].set_title("Hidden parameter trajectory during training")
axes[1].legend()

plt.tight_layout()
plt.show()


## 13. Evaluate the reconstructed trajectory on the full dense grid

The sparse observations are not the whole story.

A good inverse model should also reconstruct the trajectory **between** those observed times.  
That is why we evaluate on the dense truth grid.

We will measure:
- position RMSE
- velocity RMSE
- learned drag coefficient error

In [13]:
model.eval()
with torch.no_grad():
    pred_eval_state_norm = model(eval_t_tensor)
    pred_eval_state = denormalize_state(pred_eval_state_norm).cpu().numpy()

pred_position = pred_eval_state[:, :3]
pred_velocity = pred_eval_state[:, 3:]

true_position = truth_state[:, :3]
true_velocity = truth_state[:, 3:]

position_rmse_km = float(np.sqrt(np.mean(np.sum((pred_position - true_position) ** 2, axis=1))))
velocity_rmse_kms = float(np.sqrt(np.mean(np.sum((pred_velocity - true_velocity) ** 2, axis=1))))
learned_drag_coeff = float(torch.exp(log_drag_param).detach().cpu())

drag_abs_error = None if true_drag_coeff is None else abs(learned_drag_coeff - true_drag_coeff)

print(f"Position RMSE [km]:     {position_rmse_km:.6f}")
print(f"Velocity RMSE [km/s]:   {velocity_rmse_kms:.6f}")
print(f"Learned drag coeff:     {learned_drag_coeff:.8f}")
if true_drag_coeff is not None:
    print(f"True drag coeff:        {true_drag_coeff:.8f}")
    print(f"Absolute error:         {drag_abs_error:.8f}")


Position RMSE [km]:     8514.544635
Velocity RMSE [km/s]:   7.552021
Learned drag coeff:     0.00001182
True drag coeff:        0.00003245
Absolute error:         0.00002063


In [14]:
plt.figure(figsize=(7, 7))
plt.plot(true_position[:, 0], true_position[:, 1], label="Dense truth")
plt.plot(pred_position[:, 0], pred_position[:, 1], label="PINN reconstruction")
plt.scatter(obs_state[:, 0], obs_state[:, 1], s=18, label="Sparse noisy observations")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title("Orbit reconstruction in the x-y plane")
plt.axis("equal")
plt.legend()
plt.show()

plt.figure()
position_residual_norm = np.linalg.norm(pred_position - true_position, axis=1)
plt.plot(truth_t / 3600.0, position_residual_norm)
plt.xlabel("Time [hours]")
plt.ylabel("Position residual norm [km]")
plt.title("Reconstruction error over time")
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
labels = ["x_km", "y_km", "z_km"]
for i, ax in enumerate(axes):
    ax.plot(truth_t / 3600.0, true_position[:, i], label="Truth")
    ax.plot(truth_t / 3600.0, pred_position[:, i], label="Prediction")
    ax.scatter(obs_t / 3600.0, obs_state[:, i], s=12, label="Sparse obs" if i == 0 else None)
    ax.set_ylabel(labels[i])
    if i == 0:
        ax.legend()
axes[-1].set_xlabel("Time [hours]")
fig.suptitle("Position components over time")
plt.tight_layout()
plt.show()


## 14. Compare the learned parameter against the controlled inverse baseline from Notebook 1

Notebook 1 introduced a simple non-neural inverse baseline:
- assume the initial state is known
- scan candidate drag coefficients
- integrate the orbit for each candidate
- compare against sparse observations

That baseline is still useful.

### Why compare against it?
Because a PINN should not only be interesting.  
It should also be compared against a simpler method.

If the PINN performs competitively while offering flexibility for sparse, noisy, or partially observed data, then the project has a stronger scientific story.

In [15]:
def simulate_positions_for_drag(drag_coeff, t_eval_subset, initial_state):
    sol = solve_ivp(
        lambda t, y: orbital_dynamics(t, y, use_j2=True, use_drag=True, drag_coeff=drag_coeff),
        (float(t_eval_subset[0]), float(t_eval_subset[-1])),
        initial_state,
        t_eval=t_eval_subset,
        rtol=1e-8,
        atol=1e-8,
    )
    return sol.y[:3].T

initial_state_known = truth_state[0].copy()
candidate_drag_values = np.linspace(0.2e-5, 4.0e-5, 40)
baseline_losses = []

observed_positions = obs_state[:, :3]
for c in candidate_drag_values:
    pred_positions = simulate_positions_for_drag(c, obs_t, initial_state_known)
    mse = np.mean(np.sum((pred_positions - observed_positions) ** 2, axis=1))
    baseline_losses.append(mse)

baseline_losses = np.array(baseline_losses)
best_idx = np.argmin(baseline_losses)
baseline_best_drag = float(candidate_drag_values[best_idx])

plt.figure()
plt.plot(candidate_drag_values, baseline_losses)
if true_drag_coeff is not None:
    plt.axvline(true_drag_coeff, linestyle="--", label="True drag coeff")
plt.axvline(baseline_best_drag, linestyle="--", label="Best baseline coeff")
plt.axvline(learned_drag_coeff, linestyle="--", label="Learned PINN coeff")
plt.xlabel("Candidate drag coefficient")
plt.ylabel("Position MSE on sparse observations")
plt.title("Controlled inverse baseline vs learned PINN parameter")
plt.legend()
plt.show()

print(f"Best baseline drag coeff: {baseline_best_drag:.8f}")
print(f"Learned PINN drag coeff:  {learned_drag_coeff:.8f}")
if true_drag_coeff is not None:
    print(f"True drag coeff:          {true_drag_coeff:.8f}")


Best baseline drag coeff: 0.00003221
Learned PINN drag coeff:  0.00001182
True drag coeff:          0.00003245


## 15. Optional extension: turn off the physics loss to create a data-only neural baseline

A very clean ablation experiment for the paper is:
- train the same network architecture with `physics_weight = 0.0`
- compare its reconstruction and parameter-learning behavior against the physics-informed version

That isolates the effect of the physics constraint.

For a strong science-fair or early paper narrative, this comparison is very valuable.

The simplest way to run that experiment is:
1. copy the training section
2. change `config["physics_weight"]` to `0.0`
3. retrain from scratch
4. compare the trajectory RMSE and drag recovery

## 16. Real-data readiness check

This notebook does not yet train on real TLE or Horizons data.

That is intentional.

The project still needs the synthetic control case as the main place to test whether hidden-parameter recovery works at all.  
However, we can already inspect any standardized files from Notebook 2 and confirm that the project pipeline is ready to move in that direction later.

This is an important project-design idea:

> **solve the scientific problem in the controlled environment first, then port the workflow to the authentic data environment**

In [16]:
expected_standard_columns = [
    "dataset_name",
    "object_name",
    "source_name",
    "timestamp_utc",
    "t_sec",
    "x_km",
    "y_km",
    "z_km",
    "vx_kms",
    "vy_kms",
    "vz_kms",
]

for label, path in standardized_candidates.items():
    print("\n" + "=" * 70)
    print(label)
    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        missing = [col for col in expected_standard_columns if col not in df.columns]
        print("Missing expected columns:", missing)
        display(df.head(3))
    else:
        print("File not found.")



synthetic_standardized
File not found.

celestrak_standardized
Rows: 145
Columns: ['timestamp_utc', 'error_code', 'x_km', 'y_km', 'z_km', 'vx_kms', 'vy_kms', 'vz_kms', 'dataset_name', 'object_name', 'source_name', 't_sec', 'radius_km', 'speed_kms', 'altitude_km']
Missing expected columns: []


,timestamp_utc,error_code,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,dataset_name,object_name,source_name,t_sec,radius_km,speed_kms,altitude_km
0,2026-06-18 19:14:58.748642206+00:00,0,2558.580926,-6300.916339,0.006650,4.399826,1.792288,6.005513,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,0.0,6800.579613,7.657477,422.443313
1,2026-06-18 19:24:58.748642206+00:00,0,4439.200881,-3919.804175,3334.285923,1.628275,5.840270,4.680973,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,600.0,6796.236594,7.659735,418.100294
2,2026-06-18 19:34:58.748642206+00:00,0,4367.390262,185.451160,5198.125365,-1.858123,7.318800,1.293124,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,1200.0,6791.833121,7.660915,413.696821



horizons_standardized
Rows: 73
Columns: ['dataset_name', 'object_name', 'source_name', 'timestamp_utc', 'x_km', 'y_km', 'z_km', 'vx_kms', 'vy_kms', 'vz_kms', 't_sec', 'radius_km', 'speed_kms', 'altitude_km']
Missing expected columns: []


,dataset_name,object_name,source_name,timestamp_utc,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,t_sec,radius_km,speed_kms,altitude_km
0,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 00:00:00+00:00,144256.242111,329424.941296,31753.362095,-1.004401,0.420671,0.005566,0.0,361024.834797,1.088952,354646.698497
1,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 01:00:00+00:00,140632.368555,330921.171662,31771.627096,-1.008844,0.410559,0.004581,3600.0,360964.986153,1.089195,354586.849853
2,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 02:00:00+00:00,136992.696683,332380.908087,31786.342414,-1.013177,0.400397,0.003594,7200.0,360907.797878,1.089431,354529.661578


## 17. What this notebook accomplished

This notebook adds the first true learning result to the project.

### New capabilities added here
- the synthetic inverse-problem dataset can be loaded or regenerated robustly
- a first neural network now maps **time to orbital state**
- the network is no longer purely data-driven because it also minimizes a **physics residual**
- the hidden drag-like perturbation coefficient is now a **learnable model parameter**
- the learned parameter can be compared against both:
  - the known truth
  - the controlled inverse baseline from Notebook 1

### Why this matters scientifically
This is the first notebook where the project tests its central claim:

> sparse observations plus physics constraints can recover hidden physical information

If the drag coefficient recovery is reasonable and the trajectory reconstruction is strong, then the project has crossed from a setup stage into a real inverse-problem result.

## 18. Checklist to complete

- Run the training cell and save the final plots
- Record the final learned drag coefficient
- Record the position RMSE and velocity RMSE
- Compare the learned parameter to the Notebook 1 baseline
- Try at least one ablation by changing one of the following:
  - `physics_weight`
  - `epochs`
  - `COLLOCATION_POINTS`
  - observation sparsity in the regenerated dataset
- Write down whether the learned drag coefficient stayed stable during training
- Decide whether the next notebook should be:
  - an ablation notebook
  - a continuous PINN notebook
  - a first real-data modeling notebook

## 19. Reflection questions
1. Why is it scientifically useful to compare a PINN-style model against a simple inverse grid search baseline?
2. Why is synthetic data still necessary even after the project has started importing real orbital datasets?
3. If the model fits the data well but learns the wrong drag coefficient, what might that imply?

In [17]:
# --- Classical autodiff PINN cell for the orbital inverse problem ---
# Assumes these are already defined earlier in the notebook:
# obs_t_tensor, obs_y_tensor, colloc_t_tensor, initial_state_norm_tensor
# denormalize_state, two_body_acceleration_torch, j2_acceleration_torch,
# simple_drag_acceleration_torch, reference_speed, reference_acceleration, device

class OrbitPINN(nn.Module):
    def __init__(self, hidden_width=64, hidden_depth=3):
        super().__init__()
        layers = []
        in_dim = 1
        for _ in range(hidden_depth):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(nn.Tanh())
            in_dim = hidden_width
        layers.append(nn.Linear(in_dim, 6))
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)

pinn_model = OrbitPINN(hidden_width=64, hidden_depth=3).to(device)
log_drag_param_pinn = nn.Parameter(torch.tensor(math.log(1e-5), dtype=torch.float32, device=device))
pinn_optimizer = torch.optim.Adam(list(pinn_model.parameters()) + [log_drag_param_pinn], lr=2e-3)

def pinn_losses(model, log_drag_param):
    # Data loss on sparse observations
    pred_obs = model(obs_t_tensor)
    data_loss = torch.mean((pred_obs - obs_y_tensor) ** 2)

    # Initial condition loss
    t0 = torch.zeros((1, 1), dtype=torch.float32, device=device, requires_grad=True)
    pred_ic = model(t0)
    ic_loss = torch.mean((pred_ic - initial_state_norm_tensor) ** 2)

    # Physics loss at collocation points
    t_col = colloc_t_tensor.clone().detach().requires_grad_(True)
    y_norm = model(t_col)
    y_phys = denormalize_state(y_norm)

    grads = []
    for i in range(6):
        grad_i = torch.autograd.grad(
            y_phys[:, i:i+1],
            t_col,
            grad_outputs=torch.ones_like(y_phys[:, i:i+1]),
            create_graph=True,
            retain_graph=True
        )[0]
        grads.append(grad_i)

    dy_dt = torch.cat(grads, dim=1)

    r = y_phys[:, :3]
    v = y_phys[:, 3:]
    drag_coeff = torch.exp(log_drag_param)

    a_model = (
        two_body_acceleration_torch(r)
        + j2_acceleration_torch(r)
        + simple_drag_acceleration_torch(r, v, drag_coeff)
    )

    kinematic_residual = (dy_dt[:, :3] - v) / reference_speed
    dynamic_residual = (dy_dt[:, 3:] - a_model) / reference_acceleration

    physics_loss = torch.mean(kinematic_residual**2) + torch.mean(dynamic_residual**2)

    total_loss = 10.0 * data_loss + 30.0 * ic_loss + 1.0 * physics_loss
    return total_loss, data_loss, ic_loss, physics_loss, drag_coeff

# Short training run
history_pinn = []
for epoch in range(1, 401):
    pinn_optimizer.zero_grad()
    total_loss, data_loss, ic_loss, physics_loss, drag_coeff = pinn_losses(pinn_model, log_drag_param_pinn)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(list(pinn_model.parameters()) + [log_drag_param_pinn], max_norm=1.0)
    pinn_optimizer.step()

    if epoch % 50 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:4d} | "
            f"Total {total_loss.item():.4e} | "
            f"Data {data_loss.item():.4e} | "
            f"IC {ic_loss.item():.4e} | "
            f"Physics {physics_loss.item():.4e} | "
            f"Drag {drag_coeff.item():.6e}"
        )
        history_pinn.append([epoch, total_loss.item(), data_loss.item(), ic_loss.item(), physics_loss.item(), drag_coeff.item()])

Epoch    1 | Total 3.5472e+03 | Data 1.0169e+00 | IC 1.0655e+00 | Physics 3.5051e+03 | Drag 1.000000e-05


Epoch   50 | Total 3.5028e+01 | Data 1.1378e+00 | IC 5.5499e-01 | Physics 7.0009e+00 | Drag 9.926783e-06


Epoch  100 | Total 2.0077e+01 | Data 1.5194e+00 | IC 7.2798e-02 | Physics 2.6992e+00 | Drag 9.906072e-06


Epoch  150 | Total 1.8496e+01 | Data 1.5005e+00 | IC 7.4499e-02 | Physics 1.2562e+00 | Drag 9.904560e-06


Epoch  200 | Total 1.7569e+01 | Data 1.5293e+00 | IC 6.3008e-02 | Physics 3.8524e-01 | Drag 9.901689e-06


Epoch  250 | Total 1.7401e+01 | Data 1.5405e+00 | IC 5.9376e-02 | Physics 2.1514e-01 | Drag 9.898111e-06


Epoch  300 | Total 1.7294e+01 | Data 1.5323e+00 | IC 6.2268e-02 | Physics 1.0238e-01 | Drag 9.920763e-06


Epoch  350 | Total 1.7195e+01 | Data 1.5409e+00 | IC 5.9259e-02 | Physics 8.2438e-03 | Drag 1.006141e-05


Epoch  400 | Total 1.7211e+01 | Data 1.5511e+00 | IC 5.6059e-02 | Physics 1.8289e-02 | Drag 1.033110e-05


In [18]:
## === IMPROVED MODEL: Fourier-feature physics-informed neural network ===
# The plain tanh-MLP above suffers from spectral bias and cannot represent the
# eight high-frequency orbital cycles, so its reconstruction collapses toward the
# orbit mean (position RMSE on the order of thousands of km). Embedding the scalar
# time input in a bank of sinusoidal Fourier features lets the same small network
# represent high-frequency motion, and the physics residual is evaluated with exact
# automatic differentiation instead of finite differences.
import math, json
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42); np.random.seed(42)

# Reuse the data and torch-physics already defined above.
_t = truth_t.astype(float); _Y = truth_state.astype(float)
_ot = obs_t.astype(float); _oY = obs_state.astype(float)
_mean = _Y.mean(0, keepdims=True); _std = _Y.std(0, keepdims=True); _std[_std < 1e-12] = 1.0
_t0 = _t.min(); _tsc = _t.max() - _t.min()
_norm_t = lambda a: ((a - _t0) / _tsc).reshape(-1, 1)
_mean_T = torch.tensor(_mean, dtype=torch.float32, device=device)
_std_T = torch.tensor(_std, dtype=torch.float32, device=device)
_denorm = lambda yn: yn * _std_T + _mean_T
_ref_sp = float(np.mean(np.linalg.norm(_Y[:, 3:], axis=1)))
_ref_ac = float(MU_EARTH / (np.mean(np.linalg.norm(_Y[:, :3], axis=1)) ** 2))

K_FOURIER = 24
_freqs = torch.arange(1, K_FOURIER + 1, dtype=torch.float32, device=device).reshape(1, -1)
def fourier_features(t):
    ang = 2 * math.pi * t * _freqs
    return torch.cat([t, torch.sin(ang), torch.cos(ang)], dim=1)

class FourierPINN(nn.Module):
    def __init__(self, emb_dim, width=128, depth=3):
        super().__init__()
        layers = [nn.Linear(emb_dim, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 6)]
        self.net = nn.Sequential(*layers)
    def forward(self, emb):
        return self.net(emb)

_obs_emb = fourier_features(torch.tensor(_norm_t(_ot), dtype=torch.float32, device=device))
_obs_y = torch.tensor((_oY - _mean) / _std, dtype=torch.float32, device=device)
_y0 = torch.tensor(((_Y[0:1] - _mean) / _std), dtype=torch.float32, device=device)
_emb0 = fourier_features(torch.zeros((1, 1), dtype=torch.float32, device=device))
_colloc = torch.linspace(0, 1, 400, device=device).reshape(-1, 1)

fpinn = FourierPINN(_obs_emb.shape[1]).to(device)
log_drag_fourier = nn.Parameter(torch.tensor(math.log(1e-5), dtype=torch.float32, device=device))
_opt = torch.optim.Adam(list(fpinn.parameters()) + [log_drag_fourier], lr=2e-3)
_sched = torch.optim.lr_scheduler.StepLR(_opt, step_size=2000, gamma=0.5)

FOURIER_EPOCHS = 5000
DATA_W, IC_W, PHYS_W = 10.0, 30.0, 20.0
fourier_history = {"epoch": [], "total": [], "data": [], "ic": [], "physics": [], "drag": []}
for epoch in range(1, FOURIER_EPOCHS + 1):
    _opt.zero_grad()
    data_loss = torch.mean((fpinn(_obs_emb) - _obs_y) ** 2)
    ic_loss = torch.mean((fpinn(_emb0) - _y0) ** 2)
    tc = _colloc.clone().detach().requires_grad_(True)
    yc = _denorm(fpinn(fourier_features(tc)))
    grads = [torch.autograd.grad(yc[:, i:i+1], tc, torch.ones_like(yc[:, i:i+1]),
                                 create_graph=True, retain_graph=True)[0] for i in range(6)]
    dydt = torch.cat(grads, dim=1) / _tsc
    r, v = yc[:, :3], yc[:, 3:]
    drag_coeff = torch.exp(log_drag_fourier)
    acc = (two_body_acceleration_torch(r) + j2_acceleration_torch(r)
           + simple_drag_acceleration_torch(r, v, drag_coeff))
    physics_loss = torch.mean(((dydt[:, :3] - v) / _ref_sp) ** 2) + torch.mean(((dydt[:, 3:] - acc) / _ref_ac) ** 2)
    total = DATA_W * data_loss + IC_W * ic_loss + PHYS_W * physics_loss
    total.backward()
    torch.nn.utils.clip_grad_norm_(list(fpinn.parameters()) + [log_drag_fourier], 1.0)
    _opt.step(); _sched.step()
    if epoch % 500 == 0 or epoch == 1:
        fourier_history["epoch"].append(epoch)
        for k, val in [("total", total), ("data", data_loss), ("ic", ic_loss), ("physics", physics_loss)]:
            fourier_history[k].append(float(val.detach().cpu()))
        fourier_history["drag"].append(float(drag_coeff.detach().cpu()))
        print(f"Epoch {epoch:5d} | total={float(total):.4e} | data={float(data_loss):.3e} | "
              f"ic={float(ic_loss):.3e} | physics={float(physics_loss):.3e} | drag={float(drag_coeff):.4e}")

# ---- Evaluate reconstruction on the dense held-out grid ----
fpinn.eval()
with torch.no_grad():
    _pred = _denorm(fpinn(fourier_features(torch.tensor(_norm_t(_t), dtype=torch.float32, device=device)))).cpu().numpy()
fourier_pos_rmse = float(np.sqrt(np.mean(np.sum((_pred[:, :3] - _Y[:, :3]) ** 2, axis=1))))
fourier_vel_rmse = float(np.sqrt(np.mean(np.sum((_pred[:, 3:] - _Y[:, 3:]) ** 2, axis=1))))
fourier_drag_joint = float(torch.exp(log_drag_fourier).detach().cpu())

# ---- Frozen-trajectory least-squares drag estimate ----
# Once the reconstruction is fixed, the drag acceleration is linear in the coefficient,
# so the optimal coefficient has a closed form. Fixing the trajectory prevents the
# network from absorbing the drag signature into trajectory adjustments.
_tg = torch.tensor(_norm_t(_t), dtype=torch.float32, device=device).requires_grad_(True)
_yg = _denorm(fpinn(fourier_features(_tg)))
_grads = [torch.autograd.grad(_yg[:, i:i+1], _tg, torch.ones_like(_yg[:, i:i+1]),
                              create_graph=False, retain_graph=True)[0] for i in range(6)]
_dydt = torch.cat(_grads, dim=1) / _tsc
_r = _yg[:, :3].detach(); _v = _yg[:, 3:].detach(); _dvdt = _dydt[:, 3:].detach()
_H = 60.0
_alt = torch.clamp(torch_norm(_r) - R_EARTH, min=0.0)
_phi = -torch.exp(-_alt / _H) * torch_norm(_v) * _v               # a_drag = drag_coeff * _phi
_resid = _dvdt - two_body_acceleration_torch(_r) - j2_acceleration_torch(_r)
fourier_drag_lsq = float((torch.sum(_phi * _resid) / torch.sum(_phi * _phi)).cpu())

print("\n--- Fourier-feature PINN summary ---")
print(f"Position RMSE [km]:            {fourier_pos_rmse:.4f}")
print(f"Velocity RMSE [km/s]:         {fourier_vel_rmse:.6f}")
print(f"True drag coefficient:        {true_drag_coeff:.4e}")
print(f"Jointly-learned drag coeff:   {fourier_drag_joint:.4e}  ({100*abs(fourier_drag_joint-true_drag_coeff)/true_drag_coeff:.1f}% error)")
print(f"Frozen-LSQ drag coeff:        {fourier_drag_lsq:.4e}  ({100*abs(fourier_drag_lsq-true_drag_coeff)/true_drag_coeff:.1f}% error)")
print(f"Grid-search baseline drag:    {baseline_best_drag:.4e}  ({100*abs(baseline_best_drag-true_drag_coeff)/true_drag_coeff:.1f}% error)")

import matplotlib.pyplot as plt
# Reconstruction in xy-plane
plt.figure(figsize=(7, 7))
plt.plot(_Y[:, 0], _Y[:, 1], label="Dense truth")
plt.plot(_pred[:, 0], _pred[:, 1], "--", label="Fourier-PINN reconstruction")
plt.scatter(_oY[:, 0], _oY[:, 1], s=16, color="k", zorder=5, label="Sparse noisy obs")
plt.xlabel("x [km]"); plt.ylabel("y [km]"); plt.axis("equal")
plt.title("Fourier-feature PINN: orbit reconstruction (x-y)"); plt.legend(); plt.show()

# Reconstruction error over time
plt.figure()
plt.plot(_t / 3600.0, np.linalg.norm(_pred[:, :3] - _Y[:, :3], axis=1))
plt.xlabel("Time [hours]"); plt.ylabel("Position error [km]")
plt.title("Fourier-feature PINN reconstruction error over time"); plt.show()

# Training history + learned drag
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for k in ["total", "data", "ic", "physics"]:
    ax[0].semilogy(fourier_history["epoch"], fourier_history[k], label=k)
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].set_title("Fourier-PINN training history"); ax[0].legend()
ax[1].plot(fourier_history["epoch"], fourier_history["drag"], marker="o", label="Jointly-learned drag")
ax[1].axhline(true_drag_coeff, ls="--", color="k", label="True drag")
ax[1].axhline(fourier_drag_lsq, ls=":", color="g", label="Frozen-LSQ drag")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Drag coefficient"); ax[1].set_title("Drag coefficient estimates"); ax[1].legend()
plt.tight_layout(); plt.show()

_final = dict(
    plain_mlp_pos_rmse_km=float(position_rmse_km),
    fourier_pos_rmse_km=fourier_pos_rmse,
    fourier_vel_rmse_kms=fourier_vel_rmse,
    true_drag_coeff=float(true_drag_coeff),
    fourier_drag_joint=fourier_drag_joint,
    fourier_drag_joint_rel_pct=float(100*abs(fourier_drag_joint-true_drag_coeff)/true_drag_coeff),
    fourier_drag_lsq=fourier_drag_lsq,
    fourier_drag_lsq_rel_pct=float(100*abs(fourier_drag_lsq-true_drag_coeff)/true_drag_coeff),
    baseline_drag=float(baseline_best_drag),
    baseline_rel_pct=float(100*abs(baseline_best_drag-true_drag_coeff)/true_drag_coeff),
    n_sparse=int(len(_ot)), n_collocation=400, epochs=FOURIER_EPOCHS,
    k_fourier=K_FOURIER, phys_weight=PHYS_W,
)
json.dump(_final, open("/tmp/pinn_run/results/nb3_final_metrics.json", "w"), indent=2)
print("\nNB3_FINAL", json.dumps(_final, indent=2))


Epoch     1 | total=2.2887e+05 | data=1.009e+00 | ic=9.664e-01 | physics=1.144e+04 | drag=1.0000e-05


Epoch   500 | total=3.8744e-02 | data=1.236e-04 | ic=1.058e-03 | physics=2.878e-04 | drag=8.8232e-06


Epoch  1000 | total=1.5388e-02 | data=5.017e-05 | ic=4.303e-04 | physics=9.884e-05 | drag=8.8234e-06


Epoch  1500 | total=3.6924e-02 | data=3.371e-04 | ic=4.910e-04 | physics=9.411e-04 | drag=8.8017e-06


Epoch  2000 | total=4.0544e-03 | data=4.937e-05 | ic=4.795e-05 | physics=1.061e-04 | drag=8.8024e-06


...

Epoch  3500 | total=1.0408e-03 | data=1.673e-05 | ic=2.457e-06 | physics=3.999e-05 | drag=8.8047e-06


Epoch  4000 | total=4.4088e-05 | data=4.348e-07 | ic=6.230e-07 | physics=1.053e-06 | drag=8.8085e-06


Epoch  4500 | total=6.2516e-06 | data=8.127e-08 | ic=1.315e-11 | physics=2.719e-07 | drag=8.8085e-06


Epoch  5000 | total=5.3323e-06 | data=8.048e-08 | ic=1.140e-11 | physics=2.264e-07 | drag=8.8100e-06

--- Fourier-feature PINN summary ---
Position RMSE [km]:            0.7304
Velocity RMSE [km/s]:         0.001183
True drag coefficient:        3.2450e-05
Jointly-learned drag coeff:   8.8100e-06  (72.9% error)
Frozen-LSQ drag coeff:        2.2969e-05  (29.2% error)
Grid-search baseline drag:    3.2205e-05  (0.8% error)



NB3_FINAL {
  "plain_mlp_pos_rmse_km": 8514.544635349721,
  "fourier_pos_rmse_km": 0.7304404647226742,
  "fourier_vel_rmse_kms": 0.001183279077550703,
  "true_drag_coeff": 3.245e-05,
  "fourier_drag_joint": 8.810012332105543e-06,
  "fourier_drag_joint_rel_pct": 72.85050128781035,
  "fourier_drag_lsq": 2.296870297868736e-05,
  "fourier_drag_lsq_rel_pct": 29.218172638867923,
  "baseline_drag": 3.220512820512821e-05,
  "baseline_rel_pct": 0.7546126190193939,
  "n_sparse": 72,
  "n_collocation": 400,
  "epochs": 5000,
  "k_fourier": 24,
  "phys_weight": 20.0
}


In [19]:

import json
_m = dict(
    true_drag_coeff=float(true_drag_coeff),
    learned_drag_coeff_fd=float(learned_drag_coeff),
    drag_abs_error_fd=float(drag_abs_error),
    drag_rel_error_fd_pct=float(100*drag_abs_error/true_drag_coeff),
    position_rmse_km=float(position_rmse_km),
    velocity_rmse_kms=float(velocity_rmse_kms),
    baseline_best_drag=float(baseline_best_drag),
    baseline_rel_error_pct=float(100*abs(baseline_best_drag-true_drag_coeff)/true_drag_coeff),
    n_sparse=int(len(obs_t)),
    n_collocation=int(len(colloc_t)),
    n_dense=int(len(truth_t)),
    epochs=int(config['epochs']),
    final_total_loss=float(history_df['total_loss'].iloc[-1]),
    final_data_loss=float(history_df['data_loss'].iloc[-1]),
    final_physics_loss=float(history_df['physics_loss'].iloc[-1]),
)
try:
    _m['learned_drag_coeff_autodiff'] = float(drag_coeff.item())
    _m['drag_rel_error_autodiff_pct'] = float(100*abs(drag_coeff.item()-true_drag_coeff)/true_drag_coeff)
except Exception as e:
    _m['autodiff_error'] = str(e)
json.dump(_m, open('/tmp/pinn_run/results/nb3_metrics.json','w'), indent=2)
print('NB3_METRICS', json.dumps(_m, indent=2))


NB3_METRICS {
  "true_drag_coeff": 3.245e-05,
  "learned_drag_coeff_fd": 1.1817735867225565e-05,
  "drag_abs_error_fd": 2.0632264132774438e-05,
  "drag_rel_error_fd_pct": 63.581707651076854,
  "position_rmse_km": 8514.544635349721,
  "velocity_rmse_kms": 7.552021456535529,
  "baseline_best_drag": 3.220512820512821e-05,
  "baseline_rel_error_pct": 0.7546126190193939,
  "n_sparse": 72,
  "n_collocation": 160,
  "n_dense": 2500,
  "epochs": 600,
  "final_total_loss": 12.629952430725098,
  "final_data_loss": 1.1650595664978027,
  "final_physics_loss": 0.951999306678772,
  "learned_drag_coeff_autodiff": 8.810004146653228e-06,
  "drag_rel_error_autodiff_pct": 72.85052651262488
}


In [20]:
print('FIG_FILES', sorted(__import__('os').listdir('figs')))

FIG_FILES ['nb1_fig01.png', 'nb1_fig02.png', 'nb1_fig03.png', 'nb1_fig04.png', 'nb2_fig01.png', 'nb2_fig02.png', 'nb2_fig03.png', 'nb2_fig04.png', 'nb2_fig05.png', 'nb3_fig01.png', 'nb3_fig02.png', 'nb3_fig03.png', 'nb3_fig04.png', 'nb3_fig05.png', 'nb3_fig06.png', 'nb3_fig07.png', 'nb3_fig08.png', 'nb3_fig09.png', 'nb3_fig10.png']
